# Notebook Summary
- Test getting data with `Jikan API`
    - Character Data (Guess the Character)
    - Anime Opening (Guess the Opening / Ending)
    - Anime Title (Anidle) and other metadata

# Imports

In [132]:
import requests
import time
import random
from IPython.display import Image, display
from typing import List, Optional, Dict, Any

# Paths

In [88]:
BASE_URL = f"https://api.jikan.moe/v4"

# Utils

In [89]:
def _get(endpoint: str,
        retry_after: Optional[int] = 1,
        request_delay: Optional[float] = 0.5,
        *args, **kwargs) -> Dict[str, Any]:
    """Thin wrapper around GET with basic back-off."""
    url = f"{BASE_URL}{endpoint}"
    while True:
        resp = requests.get(url, *args, **kwargs)
        # rate-limited – wait and retry
        if resp.status_code == 429:
            time.sleep(retry_after)
            continue
        resp.raise_for_status()
        time.sleep(request_delay)            # friendly throttling
        return resp.json()

# Character Data

## GET Request

In [112]:
def test_get_characters(anime_id: int) -> dict:
    try:
        character = _get(f"/anime/{anime_id}/characters")
    except Exception as e:
        return {"error": str(e)}
    return character

In [113]:
def parse_character_data(character_data: dict, 
                        character_index: int = 0) -> dict:
    try:
        character = character_data['data'][character_index]['character']
        return character
    except KeyError:
        return "Error: No character data found"

In [114]:
anime_id = 1 # apparently this is the ID for Cowboy Bebop
character_id = 1 # this is the ID for Spike Spiegel

# sanity check
test_character = test_get_characters(anime_id)
parsed_character = parse_character_data(test_character, character_index = character_id)
image_url = parsed_character.get('images', {}).get('jpg', {}).get('image_url', '')
display(Image(url=image_url))
print(parsed_character['name'])

Spiegel, Spike


This is usable, will need the top X anime to filter the character depending on difficulty

## Get Top X Anime / Top X Characters

Design:

Will check on how we define the 'Guess the Character' part of the game. Currently there are 3 routes we can take:
- use the getUserFavorites -> take the mal_ids -> use random to randomly pick 12 character (will not think of difficulty) -> use that to construct the 12 characters
- use the getTopCharacters -> set the limit as 2000 or smth -> randomly take the 4 from top `x` for easy, then from `x to 2x` and so on
- get the Top X anime's MAL_ID -> random_choice -> filter_character using the above function -> just pick 12 [difficulty is based on the anime ranking]
- this is unlikely -> but we can combine them all (the 3 routes)

## User Favorites

In [47]:
# will skip this one first since I cannot test, will come back to it later once I have the user_id for `RPOTI`

## Get Top Characters

In [127]:
def _get_top_characters(max_items: int = 500) -> List[Dict[str, Any]]:
    """
    Collects up to `max_items` top characters.
    Endpoint: /top/characters
    """
    characters: List[Dict[str, Any]] = []
    per_page = 25
    pages_needed = (max_items + per_page - 1) // per_page
    print(pages_needed)
    for page in range(1, pages_needed + 1):
        payload = _get("/top/characters", params={"page": page, "limit": per_page})
        characters.extend(payload["data"])
        if len(characters) >= max_items or not payload["pagination"]["has_next_page"]:
            break
    return characters[:max_items]

In [130]:
def create_character_dict(max_items: int = 500,
                          range_easy: int = 100,
                          range_medium: int = 200) -> Dict[str, List[str]]:
    """
    Create a dictionary with image URLs as keys and names as values.
    """
    data = _get_top_characters(max_items)
    # make the keys as the image URLs and the values as a list of names
    # might not be the best way to do this, but it works for now
    def pick_range(start: int, end: int) -> Dict[str, List[str]]:
        subset = data[start:end]
        chosen = random.sample(subset, 4)
        return {
            character.get('images', {}).get('jpg', {}).get('image_url', ''):
                [character['name']] + character.get('nicknames', [])
            for character in chosen
        }
    return {
        "easy": pick_range(0, range_easy),
        "medium": pick_range(range_easy, range_medium),
        "hard": pick_range(range_medium, max_items)
    }

In [133]:
guess = create_character_dict()
guess_easy = guess['easy']
guess_medium = guess['medium']  
guess_hard = guess['hard']

# takes a while to run, might need to run it first before playing (but since it is once per day, it should be fine)
guess_easy

20


{'https://cdn.myanimelist.net/images/characters/14/559023.jpg': ['Erwin Smith'],
 'https://cdn.myanimelist.net/images/characters/3/174561.jpg': ['Hisoka Morow'],
 'https://cdn.myanimelist.net/images/characters/11/510227.jpg': ['Roy Mustang',
  'Flame Alchemist'],
 'https://cdn.myanimelist.net/images/characters/5/136769.jpg': ['Sanji',
  'Black Leg',
  'Mr. Prince',
  'Soba Mask']}

In [137]:
for k,v in guess_easy.items():
    display(Image(url=k))
    print(f"\n names and nicks :{v}")


 names and nicks :['Erwin Smith']



 names and nicks :['Hisoka Morow']



 names and nicks :['Roy Mustang', 'Flame Alchemist']



 names and nicks :['Sanji', 'Black Leg', 'Mr. Prince', 'Soba Mask']


note to self: 
- need to implement some sort of database if I want to make some sort of lookahead search (1 time process), top 5000 anime
    - current idea on top of mind is:
        - use sqlite, and make 2 database
            - Title
            - Characters
- Use that for lookahead search